 Preprocessing


In [90]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', 100)

## Step 1 — Load Data

In [91]:
df = pd.read_csv('../data/raw/training_v2.csv')
print(f'Shape: {df.shape}')

Shape: (91713, 186)


## Step 2 — Drop ID & Leakage Columns

- **ID columns:** encounter_id, patient_id, hospital_id carry no predictive information.
- **Leakage columns:** apache_4a_* are pre-computed death probabilities — they encode the target directly.

In [92]:
id_cols      = ['encounter_id', 'patient_id', 'hospital_id']
leakage_cols = ['apache_4a_hospital_death_prob', 'apache_4a_icu_death_prob']

df = df.drop(columns=id_cols + leakage_cols)

print(f'Shape after dropping ID and leakage columns: {df.shape}')

Shape after dropping ID and leakage columns: (91713, 181)


## Step 3 — Train / Test Split

Split before any imputation or feature selection to prevent data leakage.
- `stratify=y` ensures both sets have the same death rate (~8.6%).
- 80% train / 20% test.

In [93]:
target = 'hospital_death'

X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'X_train : {X_train.shape}')
print(f'X_test  : {X_test.shape}')
print(f'y_train — death rate: {y_train.mean():.3f}')
print(f'y_test  — death rate: {y_test.mean():.3f}')

X_train : (73370, 180)
X_test  : (18343, 180)
y_train — death rate: 0.086
y_test  — death rate: 0.086


## Step 4 — Drop High-Missing Columns

Columns with >50% missing data are dropped. Threshold is calculated **on train only** to avoid leakage.

Rationale: if more than half the values are missing, imputation would be mostly guesswork — adding noise rather than signal.

In [94]:
train_missing_pct = (X_train.isnull().sum() / len(X_train) * 100)

# Threshold analysis — to justify the 50% cutoff
for threshold in [40, 50, 60, 70]:
    n_drop = (train_missing_pct > threshold).sum()
    n_keep = X_train.shape[1] - n_drop
    print(f'Threshold >{threshold}%  →  drop {n_drop:3d} cols,  keep {n_keep:3d} cols')
print()

high_missing_cols = train_missing_pct[train_missing_pct > 50].index.tolist()

X_train = X_train.drop(columns=high_missing_cols)
X_test  = X_test.drop(columns=high_missing_cols)

print(f'Columns dropped : {len(high_missing_cols)}')
print(f'X_train shape   : {X_train.shape}')
print(f'X_test  shape   : {X_test.shape}')

Threshold >40%  →  drop  74 cols,  keep 106 cols
Threshold >50%  →  drop  74 cols,  keep 106 cols
Threshold >60%  →  drop  66 cols,  keep 114 cols
Threshold >70%  →  drop  55 cols,  keep 125 cols

Columns dropped : 74
X_train shape   : (73370, 106)
X_test  shape   : (18343, 106)


## Step 5 — Missingness Indicators

For remaining columns that still have missing values, we add a binary indicator column.

Rationale: in EDA we found that missingness is not random — certain tests are ordered only for sicker patients. The fact that a measurement was taken is itself a signal.

In [95]:
remaining_missing_cols = [c for c in X_train.columns if X_train[c].isnull().any()]

train_indicators = pd.DataFrame(
    {f'{col}_missing': X_train[col].isnull().astype(int) for col in remaining_missing_cols}
)
test_indicators = pd.DataFrame(
    {f'{col}_missing': X_test[col].isnull().astype(int) for col in remaining_missing_cols}
)

X_train = pd.concat([X_train, train_indicators], axis=1)
X_test  = pd.concat([X_test,  test_indicators],  axis=1)

print(f'Missingness indicators added: {len(remaining_missing_cols)}')
print(f'X_train shape: {X_train.shape}')
print(f'X_test  shape: {X_test.shape}')

Missingness indicators added: 99
X_train shape: (73370, 205)
X_test  shape: (18343, 205)


## Step 6 — Imputation

- **Numeric columns:** fill with train median. Median is robust to outliers (we saw extreme values in EDA).
- **Categorical columns:** fill with 'Unknown'.
- Train median is applied to both train and test — test never informs the imputation values.

In [96]:
numeric_cols     = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X_train.select_dtypes(include='object').columns.tolist()

train_medians = X_train[numeric_cols].median()

X_train[numeric_cols] = X_train[numeric_cols].fillna(train_medians)
X_test[numeric_cols]  = X_test[numeric_cols].fillna(train_medians)

X_train[categorical_cols] = X_train[categorical_cols].fillna('Unknown')
X_test[categorical_cols]  = X_test[categorical_cols].fillna('Unknown')

print(f'X_train missing: {X_train.isnull().sum().sum()}')
print(f'X_test  missing: {X_test.isnull().sum().sum()}')

X_train missing: 0
X_test  missing: 0


## Step 7 — Encoding

One-hot encoding for categorical columns. `align` ensures train and test have identical columns — any category seen in train but not in test is filled with 0.

In [97]:
X_train = pd.get_dummies(X_train, columns=categorical_cols, drop_first=False)
X_test  = pd.get_dummies(X_test,  columns=categorical_cols, drop_first=False)

X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

print(f'X_train shape after encoding: {X_train.shape}')
print(f'X_test  shape after encoding: {X_test.shape}')

X_train shape after encoding: (73370, 263)
X_test  shape after encoding: (18343, 263)


## Step 8 — Save

Save processed data to `data/processed/`. Raw data is never modified.

In [98]:
import os

os.makedirs('../data/processed', exist_ok=True)

X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv',   index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv',   index=False)

pd.Series(X_train.columns).to_csv(
    '../data/processed/feature_names.csv', index=False, header=False
)

print('Saved successfully.')
print(f'  X_train : {X_train.shape}')
print(f'  X_test  : {X_test.shape}')
print(f'  Features: {X_train.shape[1]}')

Saved successfully.
  X_train : (73370, 263)
  X_test  : (18343, 263)
  Features: 263
